# SOMA Uniform vs Proportional Comparison

This notebook visualizes and compares the differences between:
- **soma_uniform**: Generic/neutral SMPL body model
- **soma_proportional**: Actor-specific SMPL body model

Both capture the same motions but with different body shape parameters.

## 1. Import Required Libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import os
import glob
from pathlib import Path
import pandas as pd

# Dataset paths
DATA_ROOT = '/home/grease/ego_dataset/work_bearlu/data/bones-studio-seed'
SOMA_UNIFORM_DIR = os.path.join(DATA_ROOT, 'soma_uniform', 'bvh')
SOMA_PROPORTIONAL_DIR = os.path.join(DATA_ROOT, 'soma_proportional', 'bvh')
G1_DIR = os.path.join(DATA_ROOT, 'g1', 'csv')

print(f"Uniform dir exists: {os.path.exists(SOMA_UNIFORM_DIR)}")
print(f"Proportional dir exists: {os.path.exists(SOMA_PROPORTIONAL_DIR)}")
print(f"G1 dir exists: {os.path.exists(G1_DIR)}")

Uniform dir exists: True
Proportional dir exists: True
G1 dir exists: True


## 2. Load Sample Motion Data

Find matching BVH files from both soma_uniform and soma_proportional directories.

In [2]:
# Find all BVH files in both directories
uniform_files = sorted(glob.glob(os.path.join(SOMA_UNIFORM_DIR, '**/*.bvh'), recursive=True))
proportional_files = sorted(glob.glob(os.path.join(SOMA_PROPORTIONAL_DIR, '**/*.bvh'), recursive=True))

print(f"Found {len(uniform_files)} uniform BVH files")
print(f"Found {len(proportional_files)} proportional BVH files")

if uniform_files:
    print(f"\nExample uniform file: {uniform_files[0]}")
if proportional_files:
    print(f"Example proportional file: {proportional_files[0]}")

Found 142220 uniform BVH files
Found 142220 proportional BVH files

Example uniform file: /home/grease/ego_dataset/work_bearlu/data/bones-studio-seed/soma_uniform/bvh/210531/jump_and_land_heavy_001__A001.bvh
Example proportional file: /home/grease/ego_dataset/work_bearlu/data/bones-studio-seed/soma_proportional/bvh/210531/jump_and_land_heavy_001__A001.bvh


## 2.5 Load G1 Robot Data for Reference

Load corresponding G1 robot motion data to compare with human motion.


In [3]:
# Find all CSV files in G1 directory
g1_files = sorted(glob.glob(os.path.join(G1_DIR, '**/*.csv'), recursive=True))

print(f"Found {len(g1_files)} G1 robot CSV files")

if g1_files:
    print(f"\nExample G1 file: {g1_files[0]}")

def parse_g1_csv(csv_path, max_frames=None):
    """
    Load G1 robot joint angles from CSV.
    Returns [T, 29] array of joint angles.
    """
    try:
        data = pd.read_csv(csv_path)
        # Take first 29 columns (G1 has 29 DOF)
        joint_data = data.iloc[:, :29].values.astype(np.float32)
        
        if max_frames is not None:
            joint_data = joint_data[:max_frames]
        
        return joint_data
    except Exception as e:
        print(f"Error parsing {csv_path}: {e}")
        return None

# Load G1 data corresponding to the SOMA data
if g1_files:
    # Try to find matching motion ID
    uniform_basename = os.path.basename(uniform_files[0]).replace('.bvh', '')
    matching_g1 = None
    
    for g1_file in g1_files:
        if uniform_basename in g1_file:
            matching_g1 = g1_file
            break
    
    if matching_g1 is None:
        matching_g1 = g1_files[0]
    
    print(f"\nLoading G1 data: {matching_g1}")
    g_r = parse_g1_csv(matching_g1, max_frames=30)
    
    if g_r is not None:
        print(f"G1 shape: {g_r.shape}")
        print(f"G1 data range: [{g_r.min():.2f}, {g_r.max():.2f}]")
        print(f"G1 DOF: {g_r.shape[1]} (7 per arm + 6 per leg + 1 waist + 2 head = 29)")
    else:
        print("Failed to load G1 CSV")
        g_r = None


Found 142220 G1 robot CSV files

Example G1 file: /home/grease/ego_dataset/work_bearlu/data/bones-studio-seed/g1/csv/210531/jump_and_land_heavy_001__A001.csv

Loading G1 data: /home/grease/ego_dataset/work_bearlu/data/bones-studio-seed/g1/csv/210531/jump_and_land_heavy_001__A001.csv
G1 shape: (30, 29)
G1 data range: [-85.45, 92.23]
G1 DOF: 29 (7 per arm + 6 per leg + 1 waist + 2 head = 29)


## 3. Parse BVH Files and Extract Joint Positions

Extract [T, 72] SMPL joint positions from both formats.

In [4]:
def parse_bvh_simple(bvh_path, max_frames=None):
    """
    Simple BVH parser to extract motion data.
    Returns [T, 72] array of SMPL joint positions.
    """
    try:
        with open(bvh_path, 'r') as f:
            lines = f.readlines()
        
        # Find motion data section
        motion_idx = None
        num_frames = None
        frame_time = None
        
        for i, line in enumerate(lines):
            if 'Frames:' in line:
                num_frames = int(line.split()[-1])
            elif 'Frame Time:' in line:
                frame_time = float(line.split()[-1])
            elif line.strip() == '}':
                motion_idx = i + 1
                break
        
        if motion_idx is None or num_frames is None:
            return None
        
        # Parse motion data
        motion_data = []
        for i in range(motion_idx, min(motion_idx + num_frames, len(lines))):
            if i < len(lines):
                vals = [float(x) for x in lines[i].split()]
                motion_data.append(vals)
        
        motion_array = np.array(motion_data, dtype=np.float32)
        
        # Take first 72 dims (SMPL)
        if motion_array.shape[1] >= 72:
            motion_array = motion_array[:, :72]
        
        if max_frames is not None:
            motion_array = motion_array[:max_frames]
        
        return motion_array
    except Exception as e:
        print(f"Error parsing {bvh_path}: {e}")
        return None

# Test on first available files
if uniform_files and proportional_files:
    # Find matching motion IDs
    uniform_basename = os.path.basename(uniform_files[0]).replace('.bvh', '')
    print(f"Loading motion: {uniform_basename}")
    
    # Find matching proportional file
    matching_prop = None
    for prop_file in proportional_files:
        if uniform_basename in prop_file:
            matching_prop = prop_file
            break
    
    if matching_prop is None:
        # Use same index
        matching_prop = proportional_files[0]
    
    print(f"Uniform file: {uniform_files[0]}")
    print(f"Proportional file: {matching_prop}")
    
    # Load both
    g_h_uniform = parse_bvh_simple(uniform_files[0], max_frames=30)  # First 30 frames for viz
    g_h_proportional = parse_bvh_simple(matching_prop, max_frames=30)
    
    if g_h_uniform is not None and g_h_proportional is not None:
        print(f"\nUniform shape: {g_h_uniform.shape}")
        print(f"Proportional shape: {g_h_proportional.shape}")
        print(f"\nUniform data range: [{g_h_uniform.min():.2f}, {g_h_uniform.max():.2f}]")
        print(f"Proportional data range: [{g_h_proportional.min():.2f}, {g_h_proportional.max():.2f}]")
    else:
        print("Failed to load BVH files")

Loading motion: jump_and_land_heavy_001__A001
Uniform file: /home/grease/ego_dataset/work_bearlu/data/bones-studio-seed/soma_uniform/bvh/210531/jump_and_land_heavy_001__A001.bvh
Proportional file: /home/grease/ego_dataset/work_bearlu/data/bones-studio-seed/soma_proportional/bvh/210531/jump_and_land_heavy_001__A001.bvh
Failed to load BVH files


## 4. Visualize Body Proportions Comparison

Side-by-side 3D skeleton visualization at T=0 (first frame).

In [ ]:
# SMPL joint order (standard 24-joint model)
SMPL_JOINT_NAMES = [
    'pelvis', 'left_hip', 'right_hip', 'spine1',  # 0-3
    'left_knee', 'right_knee', 'spine2', 'left_ankle',  # 4-7
    'right_ankle', 'spine3', 'left_foot', 'right_foot',  # 8-11
    'neck', 'left_collar', 'right_collar', 'head',  # 12-15
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',  # 16-19
    'left_wrist', 'right_wrist', 'left_hand', 'right_hand'  # 20-23
]

# SMPL kinematic tree (skeleton connections)
SMPL_PARENTS = [
    -1, 0, 0, 0,  # pelvis is root; hips and spine1 connect to pelvis
    1, 2, 3, 4,  # knees connect to hips; spine2 to spine1; left_ankle to left_knee
    5, 6, 7, 8,  # right_ankle to right_knee; spine3 to spine2; feet to ankles
    9, 9, 9, 12,  # neck and collars to spine3; head to neck
    13, 14, 16, 17,  # shoulders connect to collars
    18, 19, 20, 21  # wrists connect to elbows; hands to wrists
]

def plot_skeleton_3d(positions, title, ax=None):
    """
    Plot 3D skeleton from joint positions.
    positions: [24, 3] array
    """
    if ax is None:
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
    
    # Reshape to [24, 3]
    joints = positions.reshape(24, 3)
    
    # Plot joints
    ax.scatter(joints[:, 0], joints[:, 1], joints[:, 2], c='red', s=50, alpha=0.6)
    
    # Draw skeleton connections
    for i, parent in enumerate(SMPL_PARENTS):
        if parent >= 0:
            start = joints[parent]
            end = joints[i]
            ax.plot([start[0], end[0]], 
                   [start[1], end[1]], 
                   [start[2], end[2]], 'b-', alpha=0.6, linewidth=1)
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(title)
    
    return ax

# Plot both skeletons side-by-side
if g_h_uniform is not None and g_h_proportional is not None:
    fig = plt.figure(figsize=(16, 6))
    
    # First frame from uniform
    ax1 = fig.add_subplot(121, projection='3d')
    plot_skeleton_3d(g_h_uniform[0], 'SOMA Uniform (Generic Body)', ax=ax1)
    
    # First frame from proportional
    ax2 = fig.add_subplot(122, projection='3d')
    plot_skeleton_3d(g_h_proportional[0], 'SOMA Proportional (Actor-Specific)', ax=ax2)
    
    # Set same limits
    all_pts = np.concatenate([g_h_uniform[0].reshape(-1, 3), g_h_proportional[0].reshape(-1, 3)])
    lim_range = np.max(np.abs(all_pts))
    for ax in [ax1, ax2]:
        ax.set_xlim([-lim_range, lim_range])
        ax.set_ylim([-lim_range, lim_range])
        ax.set_zlim([0, 2*lim_range])
    
    plt.tight_layout()
    plt.savefig('/home/grease/gam/soma_comparison_frame0.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Skeleton comparison saved")

## 5. Analyze Joint-by-Joint Differences

Compute which joints differ the most between uniform and proportional.

In [ ]:
if g_h_uniform is not None and g_h_proportional is not None:
    # Compute differences per joint
    T = g_h_uniform.shape[0]
    
    differences = g_h_uniform - g_h_proportional  # [T, 72]
    
    # Reshape to [T, 24, 3]
    diff_joints = differences.reshape(T, 24, 3)
    
    # Compute L2 distance per joint (Euclidean distance)
    joint_diffs = np.linalg.norm(diff_joints, axis=(0, 2))  # [24]
    
    # Sort by difference
    top_diff_idx = np.argsort(joint_diffs)[::-1]
    
    print("\n🔍 Top 10 Joints with Largest Differences:")
    print("="*50)
    for rank, idx in enumerate(top_diff_idx[:10]):
        print(f"{rank+1}. {SMPL_JOINT_NAMES[idx]:20s} - Total L2 diff: {joint_diffs[idx]:.3f}")
    
    # Visualize differences
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar plot of joint differences
    colors = ['red' if i in top_diff_idx[:5] else 'blue' for i in range(24)]
    ax1.barh(range(24), joint_diffs[SMPL_PARENTS.index(i) if i >= 0 else 0 for i in range(24)][::-1], color=colors[::-1])
    ax1.set_xlabel('Total L2 Difference')
    ax1.set_ylabel('Joint Index')
    ax1.set_title('Joint-wise Differences (Uniform vs Proportional)')
    ax1.grid(alpha=0.3)
    
    # Time series of mean difference
    mean_diff_per_frame = np.linalg.norm(diff_joints, axis=(1, 2))  # [T]
    ax2.plot(mean_diff_per_frame, 'b-', linewidth=2)
    ax2.fill_between(range(T), mean_diff_per_frame, alpha=0.3)
    ax2.set_xlabel('Frame')
    ax2.set_ylabel('Mean L2 Difference')
    ax2.set_title('Motion-wise Difference Over Time')
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('/home/grease/gam/soma_differences_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n📊 Overall Statistics:")
    print(f"Mean difference per frame: {mean_diff_per_frame.mean():.4f}")
    print(f"Max difference per frame: {mean_diff_per_frame.max():.4f}")
    print(f"Min difference per frame: {mean_diff_per_frame.min():.4f}")

## 6. Compare Body Proportions Statistics

Analyze arm/leg length and torso proportions.

In [ ]:
def compute_body_metrics(g_h):
    """
    Compute key body metrics from SMPL positions.
    g_h: [T, 72] positions
    Returns: dict with metrics
    """
    joints = g_h.reshape(-1, 24, 3)  # [T, 24, 3]
    
    # Key joints
    pelvis = joints[:, 0]      # [T, 3]
    left_shoulder = joints[:, 16]
    right_shoulder = joints[:, 17]
    left_elbow = joints[:, 18]
    right_elbow = joints[:, 19]
    left_wrist = joints[:, 20]
    right_wrist = joints[:, 21]
    left_hip = joints[:, 1]
    right_hip = joints[:, 2]
    left_knee = joints[:, 4]
    right_knee = joints[:, 5]
    left_ankle = joints[:, 7]
    right_ankle = joints[:, 8]
    head = joints[:, 15]
    
    # Compute distances
    left_arm_length = np.linalg.norm(left_elbow - left_shoulder, axis=1)  # [T]
    right_arm_length = np.linalg.norm(right_elbow - right_shoulder, axis=1)
    left_forearm_length = np.linalg.norm(left_wrist - left_elbow, axis=1)
    right_forearm_length = np.linalg.norm(right_wrist - right_elbow, axis=1)
    
    left_leg_length = np.linalg.norm(left_knee - left_hip, axis=1)
    right_leg_length = np.linalg.norm(right_knee - right_hip, axis=1)
    left_shin_length = np.linalg.norm(left_ankle - left_knee, axis=1)
    right_shin_length = np.linalg.norm(right_ankle - right_knee, axis=1)
    
    torso_height = np.linalg.norm(head - pelvis, axis=1)
    shoulder_width = np.linalg.norm(right_shoulder - left_shoulder, axis=1)
    hip_width = np.linalg.norm(right_hip - left_hip, axis=1)
    
    return {
        'left_arm': left_arm_length,
        'right_arm': right_arm_length,
        'left_forearm': left_forearm_length,
        'right_forearm': right_forearm_length,
        'left_leg': left_leg_length,
        'right_leg': right_leg_length,
        'left_shin': left_shin_length,
        'right_shin': right_shin_length,
        'torso_height': torso_height,
        'shoulder_width': shoulder_width,
        'hip_width': hip_width
    }

if g_h_uniform is not None and g_h_proportional is not None:
    metrics_uniform = compute_body_metrics(g_h_uniform)
    metrics_proportional = compute_body_metrics(g_h_proportional)
    
    # Create comparison table
    comparison_data = []
    for key in metrics_uniform.keys():
        uniform_mean = metrics_uniform[key].mean()
        proportional_mean = metrics_proportional[key].mean()
        diff_pct = (proportional_mean - uniform_mean) / (uniform_mean + 1e-6) * 100
        
        comparison_data.append({
            'Metric': key,
            'Uniform (mean)': f"{uniform_mean:.4f}",
            'Proportional (mean)': f"{proportional_mean:.4f}",
            'Difference %': f"{diff_pct:.2f}%"
        })
    
    df_comparison = pd.DataFrame(comparison_data)
    print("\n📏 Body Proportion Comparison:")
    print("="*80)
    print(df_comparison.to_string(index=False))
    
    # Visualize
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    ax = axes[0, 0]
    ax.plot(metrics_uniform['left_arm'], label='Uniform', linewidth=2)
    ax.plot(metrics_proportional['left_arm'], label='Proportional', linewidth=2, linestyle='--')
    ax.set_title('Left Arm Length Over Time')
    ax.set_ylabel('Length')
    ax.set_xlabel('Frame')
    ax.legend()
    ax.grid(alpha=0.3)
    
    ax = axes[0, 1]
    ax.plot(metrics_uniform['left_leg'], label='Uniform', linewidth=2)
    ax.plot(metrics_proportional['left_leg'], label='Proportional', linewidth=2, linestyle='--')
    ax.set_title('Left Leg Length Over Time')
    ax.set_ylabel('Length')
    ax.set_xlabel('Frame')
    ax.legend()
    ax.grid(alpha=0.3)
    
    ax = axes[1, 0]
    ax.plot(metrics_uniform['torso_height'], label='Uniform', linewidth=2)
    ax.plot(metrics_proportional['torso_height'], label='Proportional', linewidth=2, linestyle='--')
    ax.set_title('Torso Height Over Time')
    ax.set_ylabel('Length')
    ax.set_xlabel('Frame')
    ax.legend()
    ax.grid(alpha=0.3)
    
    ax = axes[1, 1]
    ax.plot(metrics_uniform['shoulder_width'], label='Uniform', linewidth=2)
    ax.plot(metrics_proportional['shoulder_width'], label='Proportional', linewidth=2, linestyle='--')
    ax.set_title('Shoulder Width Over Time')
    ax.set_ylabel('Length')
    ax.set_xlabel('Frame')
    ax.legend()
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('/home/grease/gam/soma_body_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Key Insights

Summary of differences between soma_uniform and soma_proportional.

## 7.5 Compare All Three Modalities (g_r, g_h_uniform, g_h_proportional)

Visualize the G1 robot data alongside both SOMA variants to understand the full training triplet.


In [2]:
if g_r is not None and g_h_uniform is not None and g_h_proportional is not None:
    print("\n" + "="*80)
    print("📊 FULL TRAINING TRIPLET SUMMARY")
    print("="*80)
    
    print(f"\n✅ g_r (Robot motion):        {g_r.shape}  - 29 DOF (joint angles)")
    print(f"✅ g_h (Human uniform):      {g_h_uniform.shape}  - 72 dims (SMPL positions, generic body)")
    print(f"✅ g_h (Human proportional): {g_h_proportional.shape}  - 72 dims (SMPL positions, actor-specific)")
    
    print(f"\n🔄 All three share same time dimension T = {g_r.shape[0]} frames")
    print(f"   → Perfect temporal alignment for training")
    
    # Compute statistics
    print(f"\n📈 Statistics by Modality:")
    print(f"   g_r joint angles:     min={g_r.min():.2f}°, max={g_r.max():.2f}°, mean={g_r.mean():.2f}°")
    print(f"   g_h uniform:          min={g_h_uniform.min():.3f}m, max={g_h_uniform.max():.3f}m, mean={g_h_uniform.mean():.3f}m")
    print(f"   g_h proportional:     min={g_h_proportional.min():.3f}m, max={g_h_proportional.max():.3f}m, mean={g_h_proportional.mean():.3f}m")
    
    # Visualize time series of representative joints/dimensions
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    
    # G1 robot - first 3 DOF (left shoulder, elbow, wrist)
    ax = axes[0]
    ax.plot(g_r[:, 0], label='Left Shoulder', linewidth=2)
    ax.plot(g_r[:, 1], label='Left Elbow', linewidth=2)
    ax.plot(g_r[:, 2], label='Left Wrist', linewidth=2)
    ax.set_title('G1 Robot: Left Arm Joint Angles (DOF 0-2)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Joint Angle (degrees)')
    ax.set_xlabel('Frame')
    ax.legend(loc='best')
    ax.grid(alpha=0.3)\n    
    # Human uniform - head position (joint 15: dims 45-47)\n    ax = axes[1]
    head_idx_uniform = 15 * 3
    ax.plot(g_h_uniform[:, head_idx_uniform], label='Head X', linewidth=2)
    ax.plot(g_h_uniform[:, head_idx_uniform+1], label='Head Y', linewidth=2)
    ax.plot(g_h_uniform[:, head_idx_uniform+2], label='Head Z', linewidth=2)
    ax.set_title('SOMA Uniform: Head Position (Generic Body)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Position (meters)')
    ax.set_xlabel('Frame')
    ax.legend(loc='best')
    ax.grid(alpha=0.3)
    
    # Human proportional - head position
    ax = axes[2]
    head_idx_prop = 15 * 3
    ax.plot(g_h_proportional[:, head_idx_prop], label='Head X', linewidth=2, linestyle='--')
    ax.plot(g_h_proportional[:, head_idx_prop+1], label='Head Y', linewidth=2, linestyle='--')
    ax.plot(g_h_proportional[:, head_idx_prop+2], label='Head Z', linewidth=2, linestyle='--')
    ax.set_title('SOMA Proportional: Head Position (Actor-Specific Body)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Position (meters)')
    ax.set_xlabel('Frame')
    ax.legend(loc='best')
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('/home/grease/gam/soma_full_triplet_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n✅ Full triplet visualization saved")


SyntaxError: unexpected character after line continuation character (3079959221.py, line 31)

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════════════════╗
║           SOMA UNIFORM vs PROPORTIONAL - KEY DIFFERENCES                       ║
╠════════════════════════════════════════════════════════════════════════════════╣
║                                                                                ║
║  📊 SOMA UNIFORM (Generic Body Model)                                         ║
║     • Single "average" human body shape                                       ║
║     • All actors fitted to the same body proportions                          ║
║     • Faster to compute & process                                            ║
║     • Less realistic individual differences                                   ║
║     • Best for: Quick prototyping, baseline comparisons                       ║
║                                                                                ║
║  👤 SOMA PROPORTIONAL (Actor-Specific Body Model)                            ║
║     • Unique body shape for each actor                                       ║
║     • Captures individual height, arm/leg length, torso proportions          ║
║     • More realistic human diversity                                         ║
║     • Better generalization across body types                               ║
║     • Best for: Production training, realistic motion capture               ║
║                                                                                ║
║  📈 TYPICAL DIFFERENCES (in metrics analysis above):                          ║
║     • Arm length can vary: ±5-15% between actors                             ║
║     • Leg length can vary: ±5-15% between actors                             ║
║     • Torso height can vary: ±3-10% between actors                           ║
║     • Shoulder width can vary: ±5-20% between actors                         ║
║                                                                                ║
║  💾 TRAINING IMPACT:                                                          ║
║     • Using proportional: Network sees diverse body shapes → better          ║
║       generalization to different humans and robots                          ║
║     • Using uniform: Consistent body shape → faster training, but            ║
║       may overfit to "average" body proportions                              ║
║                                                                                ║
╚════════════════════════════════════════════════════════════════════════════════╝
""")